In [17]:
import pandas as pd
import numpy as np

import os
import gzip
import zipfile

import pandas as pd

from data.fetcher import TickData
from data.enums import COLUMN_CONFIG
from data.utils import get_path_subset, get_ticker_path

In [4]:
df

NameError: name 'df' is not defined

In [14]:
# def fetch_okx(start_dt, end_dt, market='spot', ticker='BTCUSDT'):
#     base_dir = get_ticker_path('okx', market, ticker)
#     file_list = os.listdir(base_dir)
#     subfile_list = get_path_subset(file_list, start_dt, end_dt)
    
#     df_list = []
#     for i, path in enumerate(subfile_list, start=1):
#         print(f"[{i}/{len(subfile_list)}] Fetching Data from {path}...")
#         filepath = os.path.join(base_dir, path)
    
#         with zipfile.ZipFile(filepath) as zf:
#             filename = zf.namelist()[0]
#             with zf.open(filename) as csv_file:
#                 df = pd.read_csv(csv_file, encoding="latin-1")
#                 df.columns = ['trade_id', 'is_buyer_maker', 'qty', 'price', 'time']
#                 df = df.sort_values("time")
#                 df_list.append(df)
    
#     final_df = pd.concat(df_list)
#     final_df = final_df.rename(columns={"timestamp": 'time', "size": 'qty'})
#     final_df['is_buyer_maker'] = (final_df['is_buyer_maker'] == "buy")
#     final_df['utc_time'] = pd.to_datetime(final_df['time'], unit='ms')
#     final_df['price'] = final_df['price'].astype('float')
#     final_df['qty'] = final_df['qty'].astype('float')
#     return final_df[['utc_time', 'time', 'price', 'qty', 'is_buyer_maker']]

In [20]:
market = 'spot'
ticker = 'BONK-USDT'
start_dt = "2024-12-25"
end_dt = "2025-01-05"

tickdata = TickData()
tickdata.fetch_okx(start_dt, end_dt, "spot", 'BONK-USDT')
tickdata.fetch_okx(start_dt, end_dt, "future", 'BONK-USDT-SWAP')

[1/12] Fetching Data from BONK-USDT-trades-2024-12-25.zip...
[2/12] Fetching Data from BONK-USDT-trades-2024-12-26.zip...
[3/12] Fetching Data from BONK-USDT-trades-2024-12-27.zip...
[4/12] Fetching Data from BONK-USDT-trades-2024-12-28.zip...
[5/12] Fetching Data from BONK-USDT-trades-2024-12-29.zip...
[6/12] Fetching Data from BONK-USDT-trades-2024-12-30.zip...
[7/12] Fetching Data from BONK-USDT-trades-2024-12-31.zip...
[8/12] Fetching Data from BONK-USDT-trades-2025-01-01.zip...
[9/12] Fetching Data from BONK-USDT-trades-2025-01-02.zip...
[10/12] Fetching Data from BONK-USDT-trades-2025-01-03.zip...
[11/12] Fetching Data from BONK-USDT-trades-2025-01-04.zip...
[12/12] Fetching Data from BONK-USDT-trades-2025-01-05.zip...
[1/12] Fetching Data from BONK-USDT-SWAP-trades-2024-12-25.zip...
[2/12] Fetching Data from BONK-USDT-SWAP-trades-2024-12-26.zip...
[3/12] Fetching Data from BONK-USDT-SWAP-trades-2024-12-27.zip...
[4/12] Fetching Data from BONK-USDT-SWAP-trades-2024-12-28.zip...
[

,utc_time,time,price,qty,is_buyer_maker
245610,2024-12-24 16:00:05.008,1735056005008,0.000033,4.0,True
245611,2024-12-24 16:00:05.043,1735056005043,0.000033,1.0,True
245612,2024-12-24 16:00:05.146,1735056005146,0.000033,8.0,False
0,2024-12-24 16:00:06.179,1735056006179,0.000033,18.0,False
245613,2024-12-24 16:00:06.179,1735056006179,0.000033,1.0,False
...,...,...,...,...,...
207366,2025-01-05 15:59:56.797,1736092796797,0.000034,4.0,False
207364,2025-01-05 15:59:56.797,1736092796797,0.000034,28.0,False
103492,2025-01-05 15:59:56.797,1736092796797,0.000034,3.0,False
103487,2025-01-05 15:59:56.797,1736092796797,0.000034,140.0,False


In [145]:
def get_tick_data(df, grouper_time='utc_time'):
    df = df.copy()
    df['dolvol'] = df['price'] * df['qty']
    
    if grouper_time == 'utc_time':
        grouper = df['utc_time']
    elif grouper_time == 'time':
        grouper = df['time']
    else:
        raise ValueError()
    
    adjusted = df.groupby(grouper)[['dolvol', 'qty']].agg({"dolvol": 'sum', 'qty': 'sum'})
    adjusted = adjusted.reset_index()
    adjusted['price'] = adjusted['dolvol'] / adjusted['qty']
    adjusted['log_price'] = np.log(adjusted['price'])
    adjusted = adjusted[adjusted['price'].diff() != 0]
    adjusted['return'] = adjusted['log_price'].diff()
    adjusted['prev_time'] = adjusted[grouper_time].shift(1)
    return adjusted[[grouper_time, 'price', 'log_price', 'return', 'prev_time']].copy()

In [146]:
from hy_corr_calculator import hayashi_yoshida_estimator

tick_df = get_tick_data(tickdata['okx_spot_BONK-USDT'], 'time')
df1 = tick_df[['time', 'return']].set_index("time").copy()

tick_df = get_tick_data(tickdata['okx_future_BONK-USDT-SWAP'], 'time')
df2 = tick_df[['time', 'return']].set_index("time").copy()

# hy = hayashi_yoshida_estimator(df1, df2)

In [108]:
# df1, df2 must have time(timestamp) and return columns 
temp_df1 = df1.iloc[1:]['return']
temp_df2 = df2.iloc[1:]['return']

# Make interval
I1 = pd.IntervalIndex.from_arrays(df1.index[:-1], df1.index[1:])
I2 = pd.IntervalIndex.from_arrays(df2.index[:-1], df2.index[1:])

# Calculate covariances
hy_cov = 0
minarg = 0

i2_overlaped = []
for i2 in I2:
    # for each interval of df2
    
    found = False
    for i in range(minarg, len(I1)):
        # for each interval from previous interval's end
        
        if (I1[i].left < i2.right) and (I1[i].right >= i2.left):
            # if overlap two intervals.
            # calculate covariance
            i2_overlaped.append([I1[i], i2])
            hy_cov += temp_df1.loc[I1[i].right] * temp_df2.loc[i2.right]
            
            # set params for next iter.
            found = True
            minarg = i

        elif I1[i].left > i2.right:
            # overlap is not init. (at the Begining)
            break
        elif I1[len(I1) - 1].right <= i2.left:
            # overlap is not init (at the End)
            break
        elif found:
            # if overlap series is ended.
            break
    
# calculate HY estimator    
hy_corr = hy_cov / np.sqrt(np.sum((temp_df1) ** 2) * np.sum((temp_df2) ** 2))

In [115]:
np.sum((temp_df1) ** 2)

np.float64(0.035833360247891605)

In [125]:
np.sum((temp_df1) ** 2) * np.sum((temp_df2) ** 2)

np.float64(0.0008408600976550425)

In [150]:
k = 4
LAGS = list(range(-200, 220, 20))
base_df1 = df1[np.abs(df1['return']) > k * 10**(-4)]
base_df2 = df2[np.abs(df2['return']) > k * 10**(-4)]

for lag in LAGS:
    # print(F"LAG: {lag} HY CORRELATION IS CALCULATING ...")
    temp_df1 = base_df1.copy()
    temp_df2 = base_df2.copy()
    
    temp_df1.index = temp_df1.index + lag
    hy_corr = hayashi_yoshida_estimator(temp_df1, temp_df2)
    print(f"{lag},{hy_corr}")

-200,0.30438989522640586
-180,0.3053414274430106
-160,0.30432610368475954
-140,0.30531686080223874
-120,0.30583055005099113
-100,0.30434240897672
-80,0.3034933414591673
-60,0.3037135410659733
-40,0.3055932744804612
-20,0.30557209590666723
0,0.28586999009130176
20,0.2877009438106115
40,0.28787136290215615
60,0.2865710663337785
80,0.28684865661602943
100,0.28471850460092635
120,0.28457700697599325
140,0.28371540227187575
160,0.28335774224826094
180,0.28242285190242433
200,0.2827309847350894


In [95]:
k = 4
hy_4 = hayashi_yoshida_estimator(df1[df1['return'] > k * 10**(-4)], df2[df2['return'] > k * 10**(-4)])

In [96]:
hy_4

np.float64(1.8561542990099777)